# 🚀 VS#-15M High-Performance GPU Trainer
### Accelerated Training on Google Colab Tesla T4 GPU

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/VerScript/VS-Sharp/blob/main/colab_train.ipynb)

This notebook trains the **VS#-15M** (15 Million Parameters) deep neural language model using PyTorch and CUDA on a **Tesla T4 GPU**.

**Features:**
- 🧠 **15,000,000 Parameters**: Dynamically calculated 4-layer deep neural syntax model.
- 🌐 **Polyglot & Multi-Language**: Learns VerScript and all languages accessible via `inject` (**Python, JavaScript, C, HTML/CSS**) + conversational reasoning.
- ⚡ **Tesla T4 Acceleration**: Multi-threaded CUDA tensor cores complete epochs in seconds instead of hours.
- 🔄 **Seamless Server Integration**: Automatically exports checkpoints to `model_weights.json` for instant serving in `server.js` and PolyServer.

## 1. Verify Tesla T4 GPU Acceleration

In [ ]:
!nvidia-smi

## 2. Clone VS-Sharp Repository

In [ ]:
%cd /content
!rm -rf VS-Sharp
!git clone https://github.com/VerScript/VS-Sharp.git
%cd VS-Sharp

## 3. Verify Training Dataset
Ensure `knowledge/training_data.json` is ready. If you cloned additional VerScript org repos, you can recompile with `node build_dataset.js`.

In [ ]:
import os, json
dataset_path = 'knowledge/training_data.json'
if os.path.exists(dataset_path):
    with open(dataset_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    print(f'✅ Dataset ready with {len(data):,} prompt-response pairs!')
else:
    print('⚠️ Dataset not found, compiling...')
    !node build_dataset.js

## 4. Run 15M GPU Training Loop
Train for 50 or 100 epochs on the T4 GPU. Each epoch evaluates live LLM responses across VerScript, Python, JS, C, and Conversational tests.

In [ ]:
!python train_gpu.py --epochs 50 --batch-size 128 --samples-per-epoch 12000 --lr 0.001

## 5. Visualize Loss & Accuracy Curves

In [ ]:
import json
import matplotlib.pyplot as plt

with open('epoch_eval_history.json', 'r') as f:
    history = json.load(f)

epochs = [h['epoch'] for h in history]
losses = [h['loss'] for h in history]
accuracies = [h['accuracy'] * 100 for h in history]

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(epochs, losses, 'r-o', label='Loss')
plt.title('VS#-15M Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Cross-Entropy Loss')
plt.grid(True)
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs, accuracies, 'b-o', label='Accuracy')
plt.title('VS#-15M Training Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy (%)')
plt.grid(True)
plt.legend()

plt.tight_layout()
plt.show()

## 6. Interactive Prompt Playground
Test any custom prompt with the freshly trained 15M neural language model directly in Python.

In [ ]:
import torch
from train_gpu import VSSharp15M, generate_text, calculate_hidden_size

with open('model_weights.json', 'r') as f:
    w_data = json.load(f)

vocab = w_data['vocab']
vocab_map = {t: idx for idx, t in enumerate(vocab)}
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

hidden_size_1 = calculate_hidden_size(len(vocab))
model = VSSharp15M(len(vocab), hidden_size_1).to(device)

w = w_data['weights']
model.embedding.weight.data.copy_(torch.tensor(w['E'], dtype=torch.float32))
model.fc1.weight.data.copy_(torch.tensor(w['W1'], dtype=torch.float32).t())
model.fc1.bias.data.copy_(torch.tensor(w['b1'], dtype=torch.float32))
model.fc2.weight.data.copy_(torch.tensor(w['W2'], dtype=torch.float32).t())
model.fc2.bias.data.copy_(torch.tensor(w['b2'], dtype=torch.float32))
model.fc_out.weight.data.copy_(torch.tensor(w['W3'], dtype=torch.float32).t())
model.fc_out.bias.data.copy_(torch.tensor(w['b3'], dtype=torch.float32))

test_prompt = "write a hello world program in verscript"
print(f"Prompt: {test_prompt}")
response = generate_text(model, vocab, vocab_map, test_prompt, device, max_tokens=60, temperature=0.7)
print("\n--- Generated Neural Response ---")
print(response)

## 7. Save & Download Trained Checkpoints
Download the trained weights directly to your machine. You can download the GitHub-ready compressed archive `model_weights.json.gz` (~35 MB) or uncompressed `model_weights.json`.

In [ ]:
from google.colab import files
import os
if os.path.exists("model_weights.json.gz"):
    print("Downloading model_weights.json.gz (compressed, ~35 MB)...")
    files.download("model_weights.json.gz")
else:
    print("Downloading model_weights.json...")
    files.download("model_weights.json")


## 8. Push All Learning Directly to GitHub (Optional)
To push your trained model weights and evaluation history straight back to GitHub from Colab, supply your GitHub Personal Access Token (PAT):

In [ ]:
import getpass
github_token = getpass.getpass("Enter your GitHub Personal Access Token: ")
!git config --global user.name "VS-Sharp Trainer"
!git config --global user.email "trainer@verscript.io"
!git add model_weights.json.gz epoch_eval_history.json knowledge/training_data.json
!git commit -m "feat(weights): push trained 15M model weights and eval history from T4 GPU"
!git push https://{github_token}@github.com/VerScript/VS-Sharp.git main
print("Successfully pushed trained model weights directly to GitHub!")


## 9. Publish to Hugging Face Hub (Free 50GB Model Hosting)
Upload your trained weights to **Hugging Face Hub** for permanent, free, instant online access (up to 50 GB/file).
1. Create a free account at [huggingface.co](https://huggingface.co).
2. Get a write token from [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens).

In [ ]:
import os, getpass
from huggingface_hub import HfApi

hf_token = getpass.getpass('Enter Hugging Face Write Token: ')
hf_repo = input('Enter Hugging Face Model Repo (e.g. username/VS-Sharp-15M): ')

weights_file = 'model_weights.json.gz' if os.path.exists('model_weights.json.gz') else '/kaggle/working/model_weights.json.gz'
eval_file = 'epoch_eval_history.json' if os.path.exists('epoch_eval_history.json') else '/kaggle/working/epoch_eval_history.json'

api = HfApi(token=hf_token)
api.create_repo(repo_id=hf_repo, repo_type='model', exist_ok=True)
api.upload_file(path_or_fileobj=weights_file, path_in_repo='model_weights.json.gz', repo_id=hf_repo, repo_type='model')
api.upload_file(path_or_fileobj=eval_file, path_in_repo='epoch_eval_history.json', repo_id=hf_repo, repo_type='model')

print('✅ Published to Hugging Face Hub!')
print(f'👉 Direct Online Weights URL: https://huggingface.co/{hf_repo}/resolve/main/model_weights.json.gz')